In [ ]:
# Cell 1 - Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Cell 2 - Project paths

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm"
)

VALIDATION_CSV = (
    PROJECT_ROOT
    / "data"
    / "metadata"
    / "corpus_v1"
    / "validation.csv"
)

print(VALIDATION_CSV)
print("Exists:", VALIDATION_CSV.exists())

/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/metadata/corpus_v1/validation.csv
Exists: True


In [ ]:
# Cell 3 - Install dependencies

!pip install -q transformers datasets evaluate jiwer soundfile librosa accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 43.4 MB/s eta 0:00:00


In [ ]:
# Cell 4 - Load validation metadata

import pandas as pd

validation_df = pd.read_csv(VALIDATION_CSV)

print("Rows:", len(validation_df))
print(validation_df.head())

print(
    "Duration:",
    round(validation_df["duration_seconds"].sum() / 60, 2),
    "minutes"
)

Rows: 133
       segment_id recording_id speaker_group_id  \
0  REC090_SEG0010       REC090           SPK007   
1  REC090_SEG0011       REC090           SPK007   
2  REC090_SEG0012       REC090           SPK007   
3  REC090_SEG0013       REC090           SPK007   
4  REC090_SEG0014       REC090           SPK007   

                                          audio_path  duration_seconds  \
0  data/processed/segments/REC090/REC090_SEG0010.wav             3.280   
1  data/processed/segments/REC090/REC090_SEG0011.wav             5.200   
2  data/processed/segments/REC090/REC090_SEG0012.wav            12.048   
3  data/processed/segments/REC090/REC090_SEG0013.wav             8.080   
4  data/processed/segments/REC090/REC090_SEG0014.wav             7.160   

                                       transcription  \
0                          ssalamuɛlikum necc meryem   
1           aqay ruxxa tnayn uɛecrin sana di hulanda   
2  mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ...   
3  umi wsiɣd d

In [ ]:
# Cell 5 - GPU check

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import torch
import soundfile as sf

MODEL_NAME = "openai/whisper-small"

processor = WhisperProcessor.from_pretrained(MODEL_NAME)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16
).to("cuda")

print("Whisper loaded.")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  967MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

Whisper loaded.


In [ ]:
row = validation_df.iloc[6]

audio_file = PROJECT_ROOT / row["audio_path"]

audio, sr = sf.read(audio_file)

print("Audio:", audio_file)
print("Sample rate:", sr)
print("Reference:")
print(row["transcription"])

Audio: /content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/data/processed/segments/REC090/REC090_SEG0016.wav
Sample rate: 16000
Reference:
di lmeɣrib neccin mammec ira niɛicc


In [ ]:
inputs = processor(
    audio,
    sampling_rate=16000,
    return_tensors="pt"
)

input_features = inputs.input_features.to(
    device="cuda",
    dtype=torch.float16
)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features
    )

prediction = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("\nWhisper prediction:")
print(prediction)


Whisper prediction:
 المغريم انشين ما امشي لاني عيشة


# test whether we can push Whisper toward Latin-script transcription instead of Arabic-script output by forcing a decoding language/task configuration.

In [ ]:
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="fr",
    task="transcribe"
)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids
    )

prediction_fr = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("Reference:")
print(row["transcription"])

print("\nWhisper forced French:")
print(prediction_fr)

Reference:
di lmeɣrib neccin mammec ira niɛicc

Whisper forced French:
 D'une manière...


In [ ]:
forced_decoder_ids = processor.get_decoder_prompt_ids(
    language="es",
    task="transcribe"
)

with torch.no_grad():
    predicted_ids = model.generate(
        input_features,
        forced_decoder_ids=forced_decoder_ids
    )

prediction_es = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)[0]

print("Reference:")
print(row["transcription"])

print("\nWhisper forced Spanish:")
print(prediction_es)

Reference:
di lmeɣrib neccin mammec ira niɛicc

Whisper forced Spanish:
 de Magare.


# we can run the full 133-segment validation baseline and save every raw prediction

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
import torch
import soundfile as sf

predictions = []

for _, row in tqdm(
    validation_df.iterrows(),
    total=len(validation_df)
):

    audio_file = PROJECT_ROOT / row["audio_path"]

    audio, sr = sf.read(audio_file)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_features = inputs.input_features.to(
        device="cuda",
        dtype=torch.float16
    )

    with torch.no_grad():
      predicted_ids = model.generate(
          input_features,
          max_new_tokens=128,
          no_repeat_ngram_size=3,
          repetition_penalty=1.1
    )

    prediction = processor.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(prediction)


validation_df["whisper_raw"] = predictions

print("Finished:", len(validation_df))

  0%|          | 0/133 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Finished: 133


In [ ]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "baseline"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "whisper_small_zeroshot_validation.csv"
)

validation_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved to:")
print(OUTPUT_PATH)

Saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/baseline/whisper_small_zeroshot_validation.csv


In [ ]:
RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "baseline"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

OUTPUT_PATH = (
    RESULTS_DIR
    / "whisper_small_zeroshot_validation_norepeat.csv"
)

validation_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved to:")
print(OUTPUT_PATH)

Saved to:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/baseline/whisper_small_zeroshot_validation_norepeat.csv


In [ ]:
pd.set_option("display.max_colwidth", None)

validation_df[
    [
        "segment_id",
        "speaker_group_id",
        "transcription",
        "whisper_raw"
    ]
].head(15)

,segment_id,speaker_group_id,transcription,whisper_raw
0,REC090_SEG0010,SPK007,ssalamuɛlikum necc meryem,السلام عليكم ونشهر مريم
1,REC090_SEG0011,SPK007,aqay ruxxa tnayn uɛecrin sana di hulanda,أقى رخة 22 سنة في هولنده
2,REC090_SEG0012,SPK007,mercex ak nmis n jjiran usiɣ d zi lmeɣrib umi ira aqqay di lmeɣrib ira qqaɣas ad raḥaɣ a urupa ad ggex ad ggex maca,مرشخ وميسنجيران وصغبز المغرب ومراقع المغريبة يرقى في أوروبا
3,REC090_SEG0013,SPK007,umi wsiɣd dda ufix manayenni wa ǧi ca min ira ɣari ddhi rɛqel inu,وميوسغداء وخمنين وجشاء ميررغادي العقل
4,REC090_SEG0014,SPK007,a nec mammec ira ǧjix di lmeɣrib waǧi manayenni uffix dda,اینش ممشی رجیخ دی المغرب و جی مننیو فکد دام.
5,REC090_SEG0015,SPK007,necc di lmeɣrib ira ɣari lḥurriya inu ira ɣari yemma dd baba ɣari suctma lmuhim ɣari kulci ira teɛicex mammec nneɣni lmuhim ɛawed ayi mammec tuɣa tɛiced,نحن نشد المغرب إلى غير الحرية، ويقوم بعمل موضوع.
6,REC090_SEG0016,SPK007,di lmeɣrib neccin mammec ira niɛicc,المغريم انشين ما امشي لاني عايشة
7,REC090_SEG0017,SPK007,ag baba dd yemma wa ɣaneɣ ca ɣaneɣ ca n reḥwayej n teggit en ɣaneɣ can n reḥwayej wa ntegg ca maca waǧi iziyyan wa zeyyan ca aṭṭas am mammec ira tiɛiceɣ ag ruxa mayn aks,اكبر بديما وغان خشاء غانخ شنحواج انتجيتين غانق شن حوي جمشة وجي يزيين وزيان شات تص.
8,REC090_SEG0018,SPK007,lmuhim wsiɣd,Må hem ossert.
9,REC090_SEG0019,SPK007,nec ira ɛemmas wa wsiɣd ɣar urupa wsiɣd ddi ṭiyara jjix familya inu jjix lɛaila inu umi wsiɣd wfiɣ manyenni waǧi am mammec ira nniɣ di reɛqer inu,ونشير عمس وصغر أوروبا وصيارات جائعين والعائلين ومن يوصد في خماننا وجما مشيرا نخذ العقرين


# baseline evaluation normalization + WER/CER

In [ ]:
!pip install -q jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 53.7 MB/s eta 0:00:00


In [ ]:
import re
import unicodedata
from jiwer import wer, cer


def normalize_for_evaluation(text):
    if text is None:
        return ""

    text = unicodedata.normalize("NFC", str(text))
    text = text.lower()

    # Remove punctuation only
    text = re.sub(r'[.,!?;:"\'()\[\]{}…—–]', " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


validation_df["reference_norm"] = (
    validation_df["transcription"]
    .apply(normalize_for_evaluation)
)

validation_df["prediction_norm"] = (
    validation_df["whisper_raw"]
    .apply(normalize_for_evaluation)
)

# compute the corpus-level WER and CER:

In [ ]:
references = validation_df["reference_norm"].tolist()
predictions = validation_df["prediction_norm"].tolist()

baseline_wer = wer(references, predictions)
baseline_cer = cer(references, predictions)

print(f"WER: {baseline_wer:.4f}")
print(f"WER (%): {baseline_wer * 100:.2f}%")

print(f"CER: {baseline_cer:.4f}")
print(f"CER (%): {baseline_cer * 100:.2f}%")

WER: 1.0454
WER (%): 104.54%
CER: 0.9600
CER (%): 96.00%


compute them separately for SPK007 and SPK010

In [ ]:
for speaker in ["SPK007", "SPK010"]:

    subset = validation_df[
        validation_df["speaker_group_id"] == speaker
    ]

    speaker_wer = wer(
        subset["reference_norm"].tolist(),
        subset["prediction_norm"].tolist()
    )

    speaker_cer = cer(
        subset["reference_norm"].tolist(),
        subset["prediction_norm"].tolist()
    )

    print(f"\n{speaker}")
    print(f"Segments: {len(subset)}")
    print(f"WER: {speaker_wer * 100:.2f}%")
    print(f"CER: {speaker_cer * 100:.2f}%")


SPK007
Segments: 87
WER: 100.17%
CER: 91.53%

SPK010
Segments: 46
WER: 113.58%
CER: 105.73%


save the evaluated version


In [ ]:
EVALUATED_PATH = (
    RESULTS_DIR
    / "whisper_small_zeroshot_validation_norepeat_evaluated.csv"
)

validation_df.to_csv(
    EVALUATED_PATH,
    index=False,
    encoding="utf-8"
)

print("Saved:")
print(EVALUATED_PATH)

Saved:
/content/drive/Othercomputers/My MacBook Pro/tarifit_asr_tfm/results/baseline/whisper_small_zeroshot_validation_norepeat_evaluated.csv
